# Customer 360 & Customer Lifetime Value (CLV) Analysis

## Project Objective

This project analyzes customer purchasing behavior and calculates customer-level metrics using Python and Pandas.

Key areas of analysis:

- Customer behavior analysis
- Purchase frequency
- Revenue analysis
- Customer segmentation
- RFM analysis
- Customer Lifetime Value (CLV)

**Tools Used:** Python, Pandas, SQL Server, Power BI

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# Load datasets

customers = pd.read_csv("../data/raw/customers.csv")
orders = pd.read_csv("../data/raw/orders.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
products = pd.read_csv("../data/raw/products.csv")

In [9]:
#Verify Data
print("customers",customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)

customers (10000, 4)
Orders: (60000, 4)
Order Items: (113595, 5)
Products: (2000, 4)


In [11]:
# Display basic information about each dataset

print("CUSTOMERS")
print(customers.info())

print("\nORDERS")
print(orders.info())



CUSTOMERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   customer_id           10000 non-null  int64 
 1   customer_city         10000 non-null  object
 2   customer_state        10000 non-null  object
 3   customer_signup_date  10000 non-null  object
dtypes: int64(1), object(3)
memory usage: 312.6+ KB
None

ORDERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   order_id      60000 non-null  int64 
 1   customer_id   60000 non-null  int64 
 2   order_date    60000 non-null  object
 3   order_status  60000 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.8+ MB
None


In [12]:
print("\nORDER ITEMS")
print(order_items.info())

print("\nPRODUCTS")
print(products.info())


ORDER ITEMS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113595 entries, 0 to 113594
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   order_id    113595 non-null  int64  
 1   product_id  113595 non-null  int64  
 2   quantity    113595 non-null  int64  
 3   unit_price  113595 non-null  float64
 4   discount    113595 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 4.3 MB
None

PRODUCTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  2000 non-null   int64  
 1   category    2000 non-null   object 
 2   brand       2000 non-null   object 
 3   base_price  2000 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 62.6+ KB
None


In [13]:
# Check missing values

print("Customers Missing Values:")
print(customers.isnull().sum())

print("\nOrders Missing Values:")
print(orders.isnull().sum())

print("\nOrder Items Missing Values:")
print(order_items.isnull().sum())

print("\nProducts Missing Values:")
print(products.isnull().sum())

Customers Missing Values:
customer_id             0
customer_city           0
customer_state          0
customer_signup_date    0
dtype: int64

Orders Missing Values:
order_id        0
customer_id     0
order_date      0
order_status    0
dtype: int64

Order Items Missing Values:
order_id      0
product_id    0
quantity      0
unit_price    0
discount      0
dtype: int64

Products Missing Values:
product_id    0
category      0
brand         0
base_price    0
dtype: int64


In [14]:
# Convert date columns to datetime format

customers["customer_signup_date"] = pd.to_datetime(
    customers["customer_signup_date"]
)

orders["order_date"] = pd.to_datetime(
    orders["order_date"]
)

In [15]:
# Calculate revenue for each order item

order_items["revenue"] = (
    order_items["quantity"]
    * order_items["unit_price"]
    * (1 - order_items["discount"])
)

In [17]:
#verify
order_items.head()

,order_id,product_id,quantity,unit_price,discount,revenue
0,722028,500092,1,1289.23,0.20,1031.3840
1,722028,501558,1,4699.89,0.10,4229.9010
2,726265,501298,1,260.83,0.15,221.7055
3,726265,501082,1,565.79,0.05,537.5005
4,726265,501948,2,1977.53,0.00,3955.0600


In [18]:
# Filter delivered orders only

delivered_orders = orders[
    orders["order_status"] == "Delivered"
].copy()

print(delivered_orders.shape)

(46788, 4)


In [19]:
# Merge delivered orders with order items

customer_orders = delivered_orders.merge(
    order_items,
    on="order_id",
    how="inner"
)

print(customer_orders.shape)

customer_orders.head()

(88497, 9)


,order_id,customer_id,order_date,order_status,product_id,quantity,unit_price,discount,revenue
0,722028,102803,2024-01-01,Delivered,500092,1,1289.23,0.20,1031.3840
1,722028,102803,2024-01-01,Delivered,501558,1,4699.89,0.10,4229.9010
2,726265,100490,2024-01-01,Delivered,501298,1,260.83,0.15,221.7055
3,726265,100490,2024-01-01,Delivered,501082,1,565.79,0.05,537.5005
4,726265,100490,2024-01-01,Delivered,501948,2,1977.53,0.00,3955.0600


In [20]:
# Add customer information

customer_orders = customer_orders.merge(
    customers,
    on="customer_id",
    how="left"
)

In [21]:
# Add product information

customer_orders = customer_orders.merge(
    products,
    on="product_id",
    how="left"
)

customer_orders.head()

,order_id,customer_id,order_date,order_status,product_id,quantity,unit_price,discount,revenue,customer_city,customer_state,customer_signup_date,category,brand,base_price
0,722028,102803,2024-01-01,Delivered,500092,1,1289.23,0.20,1031.3840,Mumbai,Maharashtra,2023-03-21,Sports & Fitness,Brand_060,1245.02
1,722028,102803,2024-01-01,Delivered,501558,1,4699.89,0.10,4229.9010,Mumbai,Maharashtra,2023-03-21,Home & Kitchen,Brand_044,4498.74
2,726265,100490,2024-01-01,Delivered,501298,1,260.83,0.15,221.7055,Warangal,Telangana,2023-09-07,Electronics,Brand_078,273.18
3,726265,100490,2024-01-01,Delivered,501082,1,565.79,0.05,537.5005,Warangal,Telangana,2023-09-07,Books,Brand_008,594.35
4,726265,100490,2024-01-01,Delivered,501948,2,1977.53,0.00,3955.0600,Warangal,Telangana,2023-09-07,Grocery,Brand_081,2071.36


In [22]:
print(customer_orders.shape)
print(customer_orders.columns.tolist())

(88497, 15)
['order_id', 'customer_id', 'order_date', 'order_status', 'product_id', 'quantity', 'unit_price', 'discount', 'revenue', 'customer_city', 'customer_state', 'customer_signup_date', 'category', 'brand', 'base_price']


In [23]:
# Create customer-level metrics

customer_metrics = customer_orders.groupby("customer_id").agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("revenue", "sum"),
    total_quantity=("quantity", "sum"),
    last_purchase_date=("order_date", "max")
).reset_index()

customer_metrics.head()

,customer_id,total_orders,total_revenue,total_quantity,last_purchase_date
0,100001,1,2959.4880,2,2025-04-03
1,100002,11,72406.9750,37,2025-11-09
2,100003,2,10656.3450,7,2025-11-02
3,100004,2,5868.7390,5,2025-08-25
4,100005,2,7558.6285,5,2025-04-11


In [24]:
# Calculate Average Order Value (AOV)

customer_metrics["average_order_value"] = (
    customer_metrics["total_revenue"]
    / customer_metrics["total_orders"]
)

In [25]:
# Get the latest purchase date in the dataset

reference_date = customer_orders["order_date"].max()

# Calculate customer recency in days

customer_metrics["recency_days"] = (
    reference_date
    - customer_metrics["last_purchase_date"]
).dt.days

In [26]:
# Add customer demographic information

customer_360 = customer_metrics.merge(
    customers,
    on="customer_id",
    how="left"
)

customer_360.head()

,customer_id,total_orders,total_revenue,total_quantity,last_purchase_date,average_order_value,recency_days,customer_city,customer_state,customer_signup_date
0,100001,1,2959.4880,2,2025-04-03,2959.488000,272,Noida,Uttar Pradesh,2024-07-20
1,100002,11,72406.9750,37,2025-11-09,6582.452273,52,Madurai,Tamil Nadu,2025-01-31
2,100003,2,10656.3450,7,2025-11-02,5328.172500,59,Kolkata,West Bengal,2024-09-03
3,100004,2,5868.7390,5,2025-08-25,2934.369500,128,Kanpur,Uttar Pradesh,2024-08-21
4,100005,2,7558.6285,5,2025-04-11,3779.314250,264,Nagpur,Maharashtra,2024-12-05


In [27]:
#Final Check
print(customer_360.shape)
customer_360.head()

(8785, 10)


,customer_id,total_orders,total_revenue,total_quantity,last_purchase_date,average_order_value,recency_days,customer_city,customer_state,customer_signup_date
0,100001,1,2959.4880,2,2025-04-03,2959.488000,272,Noida,Uttar Pradesh,2024-07-20
1,100002,11,72406.9750,37,2025-11-09,6582.452273,52,Madurai,Tamil Nadu,2025-01-31
2,100003,2,10656.3450,7,2025-11-02,5328.172500,59,Kolkata,West Bengal,2024-09-03
3,100004,2,5868.7390,5,2025-08-25,2934.369500,128,Kanpur,Uttar Pradesh,2024-08-21
4,100005,2,7558.6285,5,2025-04-11,3779.314250,264,Nagpur,Maharashtra,2024-12-05


In [28]:
# Create frequency column for RFM analysis

customer_360["frequency"] = customer_360["total_orders"]

In [29]:
# Calculate RFM scores

customer_360["recency_score"] = pd.qcut(
    customer_360["recency_days"],
    5,
    labels=[5, 4, 3, 2, 1],
    duplicates="drop"
)

customer_360["frequency_score"] = pd.qcut(
    customer_360["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

customer_360["monetary_score"] = pd.qcut(
    customer_360["total_revenue"],
    5,
    labels=[1, 2, 3, 4, 5],
    duplicates="drop"
)

In [30]:
# Convert RFM scores to numeric values

customer_360["recency_score"] = customer_360["recency_score"].astype(int)
customer_360["frequency_score"] = customer_360["frequency_score"].astype(int)
customer_360["monetary_score"] = customer_360["monetary_score"].astype(int)


# Create customer segments

def customer_segment(row):

    if (
        row["recency_score"] >= 4
        and row["frequency_score"] >= 4
        and row["monetary_score"] >= 4
    ):
        return "Champions"

    elif (
        row["frequency_score"] >= 4
        and row["monetary_score"] >= 3
    ):
        return "Loyal Customers"

    elif (
        row["recency_score"] >= 4
        and row["frequency_score"] >= 3
    ):
        return "Potential Loyalists"

    elif row["recency_score"] <= 2:
        return "At Risk"

    else:
        return "Regular Customers"


customer_360["customer_segment"] = customer_360.apply(
    customer_segment,
    axis=1
)


In [31]:
customer_360["customer_segment"].value_counts()

customer_segment
At Risk                2973
Champions              1906
Regular Customers      1713
Loyal Customers        1525
Potential Loyalists     668
Name: count, dtype: int64

In [32]:
# Calculate Customer Lifetime Value (CLV)

customer_360["clv"] = (
    customer_360["average_order_value"]
    * customer_360["frequency"]
)

In [33]:
#check
customer_360[
    [
        "customer_id",
        "total_orders",
        "total_revenue",
        "average_order_value",
        "frequency",
        "clv"
    ]
].head()

,customer_id,total_orders,total_revenue,average_order_value,frequency,clv
0,100001,1,2959.4880,2959.488000,1,2959.4880
1,100002,11,72406.9750,6582.452273,11,72406.9750
2,100003,2,10656.3450,5328.172500,2,10656.3450
3,100004,2,5868.7390,2934.369500,2,5868.7390
4,100005,2,7558.6285,3779.314250,2,7558.6285


In [34]:
# Calculate customer lifespan in days within the dataset

customer_lifespan = (
    customer_orders.groupby("customer_id")["order_date"]
    .agg(["min", "max"])
    .reset_index()
)

customer_lifespan["customer_lifespan_days"] = (
    customer_lifespan["max"]
    - customer_lifespan["min"]
).dt.days

In [35]:
# Add customer lifespan information

customer_360 = customer_360.merge(
    customer_lifespan[
        ["customer_id", "customer_lifespan_days"]
    ],
    on="customer_id",
    how="left"
)

In [36]:
# Replace 0 lifespan with 1 to avoid division by zero

customer_360["customer_lifespan_days"] = (
    customer_360["customer_lifespan_days"].replace(0, 1)
)

# Calculate annualized purchase frequency

customer_360["annualized_purchase_frequency"] = (
    customer_360["frequency"]
    / customer_360["customer_lifespan_days"]
    * 365
)

# Calculate annualized CLV estimate

customer_360["annualized_clv"] = (
    customer_360["average_order_value"]
    * customer_360["annualized_purchase_frequency"]
)

In [37]:
# Create customer value tiers based on annualized customer value

customer_360["clv_segment"] = pd.qcut(
    customer_360["annualized_clv"].rank(method="first"),
    4,
    labels=["Low Value", "Medium Value", "High Value", "Top Value"]
)

In [38]:
#Check:

customer_360["clv_segment"].value_counts()

clv_segment
Low Value       2197
Medium Value    2196
High Value      2196
Top Value       2196
Name: count, dtype: int64

In [ ]:
#Final Customer 360 Dataset Check
print(customer_360.shape)

print("\nColumns:")
print(customer_360.columns.tolist())

customer_360.head()

(8785, 20)

Columns:
['customer_id', 'total_orders', 'total_revenue', 'total_quantity', 'last_purchase_date', 'average_order_value', 'recency_days', 'customer_city', 'customer_state', 'customer_signup_date', 'frequency', 'recency_score', 'frequency_score', 'monetary_score', 'customer_segment', 'clv', 'customer_lifespan_days', 'annualized_purchase_frequency', 'annualized_clv', 'clv_segment']


,customer_id,total_orders,total_revenue,total_quantity,last_purchase_date,average_order_value,recency_days,customer_city,customer_state,customer_signup_date,frequency,recency_score,frequency_score,monetary_score,customer_segment,clv,customer_lifespan_days,annualized_purchase_frequency,annualized_clv,clv_segment
0,100001,1,2959.4880,2,2025-04-03,2959.488000,272,Noida,Uttar Pradesh,2024-07-20,1,1,1,1,At Risk,2959.4880,1,365.000000,1.080213e+06,Top Value
1,100002,11,72406.9750,37,2025-11-09,6582.452273,52,Madurai,Tamil Nadu,2025-01-31,11,4,5,5,Champions,72406.9750,270,14.870370,9.788350e+04,Top Value
2,100003,2,10656.3450,7,2025-11-02,5328.172500,59,Kolkata,West Bengal,2024-09-03,2,4,1,3,Regular Customers,10656.3450,329,2.218845,1.182239e+04,Medium Value
3,100004,2,5868.7390,5,2025-08-25,2934.369500,128,Kanpur,Uttar Pradesh,2024-08-21,2,2,1,2,At Risk,5868.7390,278,2.625899,7.705359e+03,Low Value
4,100005,2,7558.6285,5,2025-04-11,3779.314250,264,Nagpur,Maharashtra,2024-12-05,2,1,1,2,At Risk,7558.6285,16,45.625000,1.724312e+05,Top Value


In [40]:
# Export final Customer 360 dataset for Power BI

customer_360.to_csv(
    "../output/customer_360_final.csv",
    index=False
)

print("Customer 360 dataset exported successfully!")

Customer 360 dataset exported successfully!
